# LeetCode 2125 — Number of Laser Beams in a Bank bằng Python

Link đề: https://leetcode.com/problems/number-of-laser-beams-in-a-bank/ · Độ khó: **Medium** · Chủ đề: Array, Math, String, Matrix

Skeleton LeetCode:

```python
class Solution:
    def numberOfBeams(self, bank: List[str]) -> int:
        ...
```

## 1. Mục tiêu học

1. Đọc kỹ điều kiện tồn tại tia laser và **chuyển lời văn thành phát biểu toán**: chỉ hai hàng *không rỗng liên tiếp* mới bắn tia cho nhau.
2. Dùng **quy tắc nhân** (nguyên lý đếm): giữa hai hàng có `a` và `b` thiết bị có đúng `a·b` tia.
3. Đi từ vét cạn theo cặp thiết bị (O((mn)²·m)) → theo cặp hàng (O(m²·n)) → quét một lượt (O(m·n)).
4. Thành thạo `str.count`, list comprehension có điều kiện, `zip(a, a[1:])` và `itertools.pairwise`.
5. Viết test đối chiếu ngẫu nhiên giữa định nghĩa gốc (brute force) và công thức rút gọn.

## 2. Đề bài & ví dụ

Ngân hàng là ma trận `m × n` cho bởi mảng chuỗi nhị phân `bank`; `bank[i]` là hàng thứ `i`, `'1'` là ô có thiết bị an ninh, `'0'` là ô trống.

Giữa hai thiết bị có **một tia laser** nếu cả hai điều kiện:
- Chúng nằm ở hai **hàng khác nhau** `r1 < r2`.
- Mọi hàng `i` với `r1 < i < r2` đều **không có thiết bị nào**.

Các tia độc lập (không chặn, không gộp nhau). Trả về **tổng số tia**.

**Ví dụ 1:** `bank = ["011001","000000","010100","001000"]` → `8`.
Hàng 0 có 3 thiết bị, hàng 1 rỗng, hàng 2 có 2, hàng 3 có 1 ⇒ `3·2 + 2·1 = 8`. Không có tia giữa hàng 0 và hàng 3 vì hàng 2 có thiết bị.

**Ví dụ 2:** `bank = ["000","111","000"]` → `0` (chỉ một hàng có thiết bị).

**Ràng buộc:** `m == bank.length`, `n == bank[i].length`, `1 ≤ m, n ≤ 500`, `bank[i][j] ∈ {'0','1'}`.

Lưu ý: vị trí **cột** của thiết bị không quan trọng — tia không cần thẳng hàng.

In [1]:
from typing import List
from itertools import pairwise, combinations
import random, time

examples = [
    (["011001", "000000", "010100", "001000"], 8),
    (["000", "111", "000"], 0),
]
for bank, exp in examples:
    print("bank:")
    for r, row in enumerate(bank):
        print(f"  hàng {r}: {row}   ({row.count('1')} thiết bị)")
    print("  kỳ vọng:", exp)

bank:
  hàng 0: 011001   (3 thiết bị)
  hàng 1: 000000   (0 thiết bị)
  hàng 2: 010100   (2 thiết bị)
  hàng 3: 001000   (1 thiết bị)
  kỳ vọng: 8
bank:
  hàng 0: 000   (0 thiết bị)
  hàng 1: 111   (3 thiết bị)
  hàng 2: 000   (0 thiết bị)
  kỳ vọng: 0


## 3. Kiến thức nền

### 3.1 `str.count` — đếm ký tự trong chuỗi

`row.count("1")` duyệt chuỗi một lần, O(n), viết bằng C nên rất nhanh.

In [2]:
row = "011001"
print(row.count("1"), row.count("0"), "010100".count("1"))

3 3 2


### 3.2 Quy tắc nhân

Nếu có `a` cách chọn phần tử thứ nhất và `b` cách chọn phần tử thứ hai (độc lập) thì có `a·b` cặp. Ở đây: chọn 1 thiết bị ở hàng trên (`a` cách) và 1 thiết bị ở hàng dưới (`b` cách) ⇒ `a·b` tia.

In [3]:
top = ["A", "B", "C"]; bottom = ["x", "y"]
pairs = [(t, b) for t in top for b in bottom]
print(pairs, "->", len(pairs), "= 3 * 2")

[('A', 'x'), ('A', 'y'), ('B', 'x'), ('B', 'y'), ('C', 'x'), ('C', 'y')] -> 6 = 3 * 2


### 3.3 Duyệt các cặp liền kề: `zip(a, a[1:])` và `itertools.pairwise`

`pairwise([3, 2, 1])` sinh `(3, 2), (2, 1)`. `pairwise` có từ Python 3.10 (LeetCode hiện dùng 3.11+).

In [4]:
a = [3, 2, 1]
print(list(zip(a, a[1:])))
print(list(pairwise(a)))
print(sum(x * y for x, y in pairwise(a)))

[(3, 2), (2, 1)]
[(3, 2), (2, 1)]
8


### 3.4 Toán tử walrus `:=`

`if (c := row.count("1")) > 0:` vừa gán vừa kiểm tra — lời giải doocs dùng cú pháp này. Tương đương hai dòng `c = row.count("1")` rồi `if c > 0:`.

In [5]:
for row in ["000", "101"]:
    if (c := row.count("1")) > 0:
        print(row, "có", c, "thiết bị")
    else:
        print(row, "rỗng")

000 rỗng
101 có 2 thiết bị


## 4. Ý tưởng nền tảng / trực giác

Liệt kê thử các tia ở ví dụ 1. Thiết bị ở hàng 0: cột 1, 2, 5. Hàng 2: cột 1, 3. Hàng 3: cột 2.

- Hàng 0 ↔ hàng 2: hàng 1 ở giữa rỗng ⇒ mọi cặp đều có tia: 3 × 2 = **6** tia.
- Hàng 2 ↔ hàng 3: liền kề ⇒ 2 × 1 = **2** tia.
- Hàng 0 ↔ hàng 3: hàng 2 ở giữa có thiết bị ⇒ **0** tia.

Quan sát:
1. Hàng **rỗng** không chặn tia và không tạo tia — có thể **xóa** khỏi bài toán.
2. Sau khi xóa hàng rỗng, chỉ **hai hàng liên tiếp** (trong danh sách còn lại) mới có tia với nhau.
3. Số tia giữa hai hàng liên tiếp = tích số thiết bị.

⇒ Đáp án = Σ `c[k] · c[k+1]` trên dãy `c` là số thiết bị của các hàng **không rỗng**, theo thứ tự.

**Góc nhìn đồ thị:** coi mỗi thiết bị là một đỉnh, mỗi tia là một cạnh. Đồ thị gồm các "tầng" (hàng không rỗng), và giữa hai tầng kề nhau là một **đồ thị hai phía đầy đủ** `K_{a,b}` có đúng `a·b` cạnh. Không có cạnh nào nhảy qua tầng. Đếm số cạnh = cộng số cạnh của từng `K_{a,b}`.

Ta liệt kê cụ thể từng tia của ví dụ 1 để đối chiếu với lời giải thích của đề:

In [6]:
bank = examples[0][0]
rows_nz = [r for r, row in enumerate(bank) if "1" in row]
beams = []
for r1, r2 in pairwise(rows_nz):
    for c1, ch1 in enumerate(bank[r1]):
        for c2, ch2 in enumerate(bank[r2]):
            if ch1 == ch2 == "1":
                beams.append(f"bank[{r1}][{c1}] -- bank[{r2}][{c2}]")
print(len(beams), "tia:")
for b in beams:
    print("  ", b)

8 tia:
   bank[0][1] -- bank[2][1]
   bank[0][1] -- bank[2][3]
   bank[0][2] -- bank[2][1]
   bank[0][2] -- bank[2][3]
   bank[0][5] -- bank[2][1]
   bank[0][5] -- bank[2][3]
   bank[2][1] -- bank[3][2]
   bank[2][3] -- bank[3][2]


In [7]:
bank = examples[0][0]
devices = [(r, col) for r, row in enumerate(bank) for col, ch in enumerate(row) if ch == "1"]
print("thiết bị:", devices)
counts = [row.count("1") for row in bank]
nonzero = [c for c in counts if c > 0]
print("số thiết bị mỗi hàng:", counts, "-> bỏ hàng rỗng:", nonzero)
print("đáp án:", " + ".join(f"{a}*{b}" for a, b in pairwise(nonzero)), "=", sum(a * b for a, b in pairwise(nonzero)))

thiết bị: [(0, 1), (0, 2), (0, 5), (2, 1), (2, 3), (3, 2)]
số thiết bị mỗi hàng: [3, 0, 2, 1] -> bỏ hàng rỗng: [3, 2, 1]
đáp án: 3*2 + 2*1 = 8


## 5. Cách 1 — Brute Force / baseline: kiểm tra từng cặp thiết bị theo đúng định nghĩa

**Ý tưởng:** liệt kê mọi thiết bị; với mỗi cặp `(r1, c1)`, `(r2, c2)` có `r1 < r2`, kiểm tra mọi hàng ở giữa có rỗng không. Đây là cài đặt **sát định nghĩa nhất** — dùng làm "chuẩn vàng" khi test.

In [8]:
def brute_devices(bank):
    devices = [(r, c) for r, row in enumerate(bank) for c, ch in enumerate(row) if ch == "1"]
    beams = 0
    for (r1, c1), (r2, c2) in combinations(devices, 2):
        if r1 == r2:
            continue
        lo, hi = min(r1, r2), max(r1, r2)
        if all("1" not in bank[i] for i in range(lo + 1, hi)):
            beams += 1
    return beams

for bank, exp in examples:
    got = brute_devices(bank)
    print(got, "OK" if got == exp else "SAI")

8 OK
0 OK


**Độ phức tạp:** gọi `D` = số thiết bị (≤ m·n = 250 000). Số cặp O(D²) và mỗi cặp kiểm tra tới O(m·n) ⇒ có thể tới 10¹⁵ phép: hoàn toàn không khả thi ở kích thước lớn.

### 5.1 Baseline tốt hơn: theo cặp **hàng**

Tia giữa hai hàng không phụ thuộc cột ⇒ chỉ cần xét cặp hàng `(r1, r2)`: nếu cả hai không rỗng và mọi hàng giữa rỗng thì cộng `c[r1]·c[r2]`. O(m²) cặp × O(m) kiểm tra (dùng `counts` đã đếm sẵn) ⇒ O(m·n + m³).

**Nút thắt:** với mỗi `r1`, chỉ có **đúng một** `r2` hợp lệ (hàng không rỗng kế tiếp) — ta tốn công thử mọi `r2` còn lại.

In [9]:
def brute_rows(bank):
    counts = [row.count("1") for row in bank]
    m = len(bank); total = 0
    for r1 in range(m):
        for r2 in range(r1 + 1, m):
            if counts[r1] and counts[r2] and all(counts[i] == 0 for i in range(r1 + 1, r2)):
                total += counts[r1] * counts[r2]
    return total

for bank, exp in examples:
    print(brute_rows(bank), "OK" if brute_rows(bank) == exp else "SAI")

8 OK
0 OK


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** với mỗi hàng không rỗng `r1`, hàng duy nhất nó bắn tia **xuống** là hàng không rỗng **gần nhất phía dưới**. Vậy chỉ cần nhớ "số thiết bị của hàng không rỗng gần nhất phía trên" khi quét từ trên xuống.

Quy tắc khi quét từng hàng với `cur = row.count('1')`:
- **Nếu** `cur == 0` **thì** bỏ qua (không đổi `pre`, không cộng gì) — hàng rỗng "trong suốt".
- **Nếu** `cur > 0` **thì** `ans += pre * cur`, rồi `pre = cur`.

**Bất biến:** sau khi xử lý hàng `i`, `pre` = số thiết bị của hàng không rỗng cuối cùng trong `bank[0..i]` (hoặc 0 nếu chưa có), và `ans` = số tia giữa các hàng trong `bank[0..i]`.

Khởi tạo `pre = 0` rất tiện: hàng không rỗng đầu tiên cộng `0 · cur = 0`.

**Vì sao không cần nhìn xa hơn một hàng?** Giả sử hàng `r1` không rỗng và hàng không rỗng kế tiếp là `r2`. Mọi hàng `r3 > r2` đều có `r2` nằm giữa `r1` và `r3` ⇒ bị chặn. Vậy mỗi hàng không rỗng chỉ "nói chuyện" với tối đa hai hàng: hàng không rỗng ngay trên và ngay dưới. Quét từ trên xuống, mỗi cặp được tính đúng một lần — tại thời điểm gặp hàng dưới.

**Kiểm tra bằng số:** ví dụ 1 có `c = [3, 2, 1]` (đã bỏ hàng rỗng). Các cặp kề: (3, 2), (2, 1) ⇒ 6 + 2 = 8. Cặp (3, 1) không kề ⇒ không tính.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Lọc hàng không rỗng rồi nhân các cặp liền kề

Hai bước rõ ràng: (1) `counts` các hàng không rỗng; (2) cộng tích các cặp kề nhau.

In [10]:
def filter_pairwise(bank):
    nz = [c for row in bank if (c := row.count("1"))]
    return sum(a * b for a, b in zip(nz, nz[1:]))

for bank, exp in examples:
    print(filter_pairwise(bank), "OK" if filter_pairwise(bank) == exp else "SAI")

8 OK
0 OK


In [11]:
class SolutionFilter:
    def numberOfBeams(self, bank: List[str]) -> int:
        nz = [c for c in map(lambda r: r.count("1"), bank) if c]
        return sum(a * b for a, b in pairwise(nz))

print([SolutionFilter().numberOfBeams(b) for b, _ in examples])

[8, 0]


**Độ phức tạp cách 2:** O(m·n) để đếm, O(m) để cộng ⇒ **O(m·n)** thời gian; O(m) bộ nhớ cho danh sách `nz`.

### 7.2 Cách 3 — Quét một lượt với biến `pre` (O(1) bộ nhớ phụ)

Cài đặt trực tiếp quy tắc ở mục 6.

In [12]:
def one_pass(bank):
    ans = pre = 0
    for row in bank:
        cur = row.count("1")
        if cur:
            ans += pre * cur
            pre = cur
    return ans

for bank, exp in examples:
    print(one_pass(bank), "OK" if one_pass(bank) == exp else "SAI")

8 OK
0 OK


**Độ phức tạp cách 3:** **O(m·n)** thời gian (đọc mỗi ô một lần — bắt buộc, vì ta phải xem hết input), **O(1)** bộ nhớ phụ.

### 7.3 Cách 4 — Đếm bằng vòng lặp tay (không dùng `str.count`)

Dành cho người học muốn thấy rõ việc đếm, hoặc khi input là ma trận số nguyên thay vì chuỗi. Có thêm tối ưu nhỏ: **không cần** chuyển chuỗi sang list.

In [13]:
def manual_count(bank):
    ans = pre = 0
    for row in bank:
        cur = 0
        for ch in row:
            if ch == "1":
                cur += 1
        if cur > 0:
            ans += pre * cur
            pre = cur
    return ans

for bank, exp in examples:
    print(manual_count(bank), "OK" if manual_count(bank) == exp else "SAI")

8 OK
0 OK


Cùng độ phức tạp O(m·n) nhưng chậm hơn `str.count` vài chục lần trong CPython vì vòng lặp Python thuần. Ta sẽ đo ở mục 11.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1. Bảng trạng thái:

| hàng | chuỗi | cur | hành động | ans | pre |
|---|---|---|---|---|---|
| — | — | — | khởi tạo | 0 | 0 |
| 0 | 011001 | 3 | ans += 0·3 | 0 | 3 |
| 1 | 000000 | 0 | bỏ qua | 0 | 3 |
| 2 | 010100 | 2 | ans += 3·2 | 6 | 2 |
| 3 | 001000 | 1 | ans += 2·1 | 8 | 1 |

In [14]:
def trace_one_pass(bank):
    ans = pre = 0
    print(f"{'hàng':>4} {'chuỗi':<10} {'cur':>3}  {'hành động':<16} {'ans':>4} {'pre':>4}")
    for i, row in enumerate(bank):
        cur = row.count("1")
        if cur:
            act = f"ans += {pre}*{cur}"
            ans += pre * cur; pre = cur
        else:
            act = "bỏ qua (rỗng)"
        print(f"{i:>4} {row:<10} {cur:>3}  {act:<16} {ans:>4} {pre:>4}")
    return ans

print("Kết quả:", trace_one_pass(examples[0][0]))

hàng chuỗi      cur  hành động         ans  pre
   0 011001       3  ans += 0*3          0    3
   1 000000       0  bỏ qua (rỗng)       0    3
   2 010100       2  ans += 3*2          6    2
   3 001000       1  ans += 2*1          8    1
Kết quả: 8


Trace thêm một ví dụ có **nhiều hàng rỗng liên tiếp** và hàng rỗng ở đầu/cuối:

In [15]:
print("Kết quả:", trace_one_pass(["000", "101", "000", "000", "111", "000", "010", "000"]))

hàng chuỗi      cur  hành động         ans  pre
   0 000          0  bỏ qua (rỗng)       0    0
   1 101          2  ans += 0*2          0    2
   2 000          0  bỏ qua (rỗng)       0    2
   3 000          0  bỏ qua (rỗng)       0    2
   4 111          3  ans += 2*3          6    3
   5 000          0  bỏ qua (rỗng)       6    3
   6 010          1  ans += 3*1          9    1
   7 000          0  bỏ qua (rỗng)       9    1
Kết quả: 9


## 9. Code nộp LeetCode

In [16]:
from typing import List

class Solution:
    def numberOfBeams(self, bank: List[str]) -> int:
        ans = 0
        pre = 0                      # số thiết bị của hàng không rỗng gần nhất phía trên
        for row in bank:
            cur = row.count("1")
            if cur > 0:
                ans += pre * cur
                pre = cur
        return ans

In [17]:
sol = Solution()
for bank, exp in examples:
    got = sol.numberOfBeams(bank)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

8 | kỳ vọng 8 | OK
0 | kỳ vọng 0 | OK


## 10. Vì sao thuật toán đúng

Gọi `R = [r_1 < r_2 < … < r_t]` là các hàng không rỗng, `c_k` là số thiết bị ở hàng `r_k`.

**Bổ đề 1:** hai thiết bị có tia ⇔ chúng nằm ở hai hàng `r_k`, `r_{k+1}` liên tiếp trong `R`.
- (⇐) Giữa `r_k` và `r_{k+1}` không có hàng không rỗng nào (theo định nghĩa "liên tiếp") ⇒ điều kiện 2 thỏa.
- (⇒) Nếu hai thiết bị ở hàng `r_a < r_b` với `b ≥ a + 2` thì hàng `r_{a+1}` nằm giữa và có thiết bị ⇒ không có tia. Hai thiết bị cùng hàng ⇒ không có tia (điều kiện 1).

**Bổ đề 2:** giữa `r_k` và `r_{k+1}` có đúng `c_k · c_{k+1}` tia (quy tắc nhân, mọi cặp đều thỏa).

⇒ Đáp án = `Σ_{k=1}^{t−1} c_k · c_{k+1}`.

**Cách 3 tính đúng tổng này:** theo bất biến ở mục 6, khi gặp hàng `r_{k+1}`, `pre = c_k` nên ta cộng đúng `c_k · c_{k+1}`; hàng `r_1` cộng `0 · c_1 = 0`; hàng rỗng không cộng gì và không đổi `pre`. ∎

## 11. Phân tích Time / Space Complexity

`m` = số hàng, `n` = số cột, `D` = số thiết bị.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Cặp thiết bị | O(D² · m·n) | O(D) |
| 1b. Cặp hàng | O(m·n + m³) | O(m) |
| 2. Lọc + pairwise | O(m·n) | O(m) |
| 3. Một lượt | O(m·n) | O(1) |
| 4. Đếm tay | O(m·n) (hằng số lớn) | O(1) |

Không thể tốt hơn O(m·n) vì phải đọc từng ô (một thiết bị bị bỏ sót có thể làm thay đổi đáp án — ví dụ biến một hàng rỗng thành không rỗng).

**Vì sao `str.count` nhanh hơn vòng lặp tay?** Cả hai đều O(n) mỗi hàng, nhưng `count` chạy trong mã C, không phải tạo đối tượng Python cho từng ký tự và không phải thông dịch từng lệnh `if`. Khác biệt hằng số thường là 20–50 lần.

Giá trị đáp án lớn nhất: 500 hàng đầy thiết bị ⇒ `499 · 500 · 500 ≈ 1,25·10⁸` — vừa `int32`, Python thì không lo tràn.

Đo trên input lớn nhất (500 × 500):

In [18]:
rng = random.Random(2125)
big = ["".join(rng.choice("0001") for _ in range(500)) for _ in range(500)]
for name, f in [("brute_rows", brute_rows), ("filter_pairwise", filter_pairwise),
                ("one_pass", one_pass), ("manual_count", manual_count)]:
    t = time.perf_counter(); r = f(big); dt = time.perf_counter() - t
    print(f"{name:<16} -> {r}  {dt*1000:.1f} ms")

brute_rows       -> 7800213  21.8 ms
filter_pairwise  -> 7800213  0.5 ms
one_pass         -> 7800213  0.5 ms
manual_count     -> 7800213  5.9 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [19]:
ALL = {"brute_devices": brute_devices, "brute_rows": brute_rows, "filter_pairwise": filter_pairwise,
       "SolutionFilter": SolutionFilter().numberOfBeams, "one_pass": one_pass,
       "manual_count": manual_count, "Solution": Solution().numberOfBeams}
edge = examples + [
    (["1"], 0),                          # 1 hàng
    (["0"], 0),
    (["1", "1"], 1),
    (["11", "00", "11"], 4),             # hàng rỗng ở giữa không chặn
    (["11", "01", "11"], 4),             # 2*1 + 1*2
    (["0", "0", "1", "0", "1", "0"], 1),
    (["111", "111", "111"], 18),
    (["1" * 500, "1" * 500], 250000),
]
for bank, exp in edge:
    for name, f in ALL.items():
        if name == "brute_devices" and sum(r.count("1") for r in bank) > 60:
            continue                      # brute force chỉ chạy trên input nhỏ
        assert f(bank) == exp, (name, bank, f(bank), exp)
print("Edge case OK:", len(edge))

Edge case OK: 10


### 12.2 Đối chiếu ngẫu nhiên với định nghĩa gốc (seed cố định)

In [20]:
rng = random.Random(42)
for _ in range(1500):
    m, n = rng.randint(1, 7), rng.randint(1, 6)
    p = rng.choice([0.05, 0.2, 0.5, 0.9])
    bank = ["".join("1" if rng.random() < p else "0" for _ in range(n)) for _ in range(m)]
    ref = brute_devices(bank)
    for name, f in ALL.items():
        assert f(bank) == ref, (name, bank)
print("1500 case ngẫu nhiên OK")

1500 case ngẫu nhiên OK


### 12.3 Tính chất: hoán vị cột không đổi đáp án; chèn hàng rỗng không đổi đáp án

Đây là kiểu test "metamorphic": không cần biết đáp án đúng, chỉ cần biết một phép biến đổi input **không được** làm đổi output. Rất hữu ích khi không có brute force.

In [21]:
rng = random.Random(7)
for _ in range(300):
    m, n = rng.randint(1, 10), rng.randint(1, 10)
    bank = ["".join(rng.choice("01") for _ in range(n)) for _ in range(m)]
    base = Solution().numberOfBeams(bank)
    perm = list(range(n)); rng.shuffle(perm)
    shuffled = ["".join(row[j] for j in perm) for row in bank]
    assert Solution().numberOfBeams(shuffled) == base
    k = rng.randint(0, m)
    assert Solution().numberOfBeams(bank[:k] + ["0" * n] + bank[k:]) == base
print("tính chất bất biến OK")

tính chất bất biến OK


### 12.4 Input lớn nhất: mọi ô đều là thiết bị

In [22]:
full = ["1" * 500] * 500
t = time.perf_counter(); r = Solution().numberOfBeams(full); dt = time.perf_counter() - t
assert r == 499 * 500 * 500 == filter_pairwise(full)
print(r, f"{dt*1000:.2f} ms")

124750000 0.14 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cập nhật `pre` cả khi hàng rỗng

In [23]:
def wrong_update_on_empty(bank):
    ans = pre = 0
    for row in bank:
        cur = row.count("1")
        ans += pre * cur
        pre = cur                       # SAI: hàng rỗng xóa mất pre
    return ans

print("bản sai :", wrong_update_on_empty(examples[0][0]))
print("bản đúng:", Solution().numberOfBeams(examples[0][0]))

bản sai : 2
bản đúng: 8


**Vì sao sai:** hàng 1 rỗng đặt `pre = 0`, nên 6 tia giữa hàng 0 và hàng 2 bị mất. Đề nói hàng rỗng **không chặn** tia. **Sửa:** chỉ cập nhật `pre` khi `cur > 0`.

### Lỗi 2 — Cộng dồn với **mọi** hàng phía trên (quên điều kiện "không có thiết bị ở giữa")

In [24]:
def wrong_all_above(bank):
    ans = above = 0
    for row in bank:
        cur = row.count("1")
        ans += above * cur
        above += cur                     # SAI: cộng dồn mọi hàng phía trên
    return ans

print("bản sai :", wrong_all_above(examples[0][0]))
print("bản đúng:", Solution().numberOfBeams(examples[0][0]))

bản sai : 11
bản đúng: 8


**Vì sao sai:** tính cả tia giữa hàng 0 và hàng 3 (3 tia), nhưng hàng 2 có thiết bị nên chặn. **Sửa:** chỉ nhân với hàng không rỗng **gần nhất**.

### Lỗi 3 — Đếm theo cột/ yêu cầu thẳng hàng

In [25]:
def wrong_same_column(bank):
    m, n = len(bank), len(bank[0]); ans = 0
    for c in range(n):                   # SAI: nghĩ tia phải thẳng đứng cùng cột
        last = None
        for r in range(m):
            if bank[r][c] == "1":
                if last is not None: ans += 1
                last = r
    return ans

print("bản sai :", wrong_same_column(examples[0][0]))
print("bản đúng:", Solution().numberOfBeams(examples[0][0]))

bản sai : 2
bản đúng: 8


**Vì sao sai:** đề không yêu cầu cùng cột; ví dụ có tia `bank[0][1] — bank[2][3]`. **Sửa:** cột không quan trọng — chỉ đếm số thiết bị mỗi hàng.

### Lỗi 4 — Cộng thay vì nhân

In [26]:
def wrong_add(bank):
    nz = [r.count("1") for r in bank if "1" in r]
    return sum(a + b for a, b in pairwise(nz))      # SAI

print("ví dụ 1 - bản sai :", wrong_add(examples[0][0]), "| bản đúng:", Solution().numberOfBeams(examples[0][0]),
      "(trùng hợp: 3+2+2+1 = 3*2+2*1 = 8!)")
b = ["111", "000", "111"]
print("phản ví dụ - bản sai :", wrong_add(b), "| bản đúng:", Solution().numberOfBeams(b))

ví dụ 1 - bản sai : 8 | bản đúng: 8 (trùng hợp: 3+2+2+1 = 3*2+2*1 = 8!)
phản ví dụ - bản sai : 6 | bản đúng: 9


Chú ý: ví dụ 1 của đề tình cờ cho cùng đáp án 8 với cả hai công thức — một lời cảnh báo rằng **qua được ví dụ của đề chưa chắc đã đúng**; luôn tự dựng thêm phản ví dụ.

**Sửa:** số **cặp** giữa hai tập kích thước `a`, `b` là `a·b` (quy tắc nhân), không phải `a + b`.

### Lỗi 5 — Đếm cả chính hàng đó (tia trong cùng một hàng)

In [27]:
def wrong_same_row(bank):
    ans = pre = 0
    for row in bank:
        cur = row.count("1")
        if cur:
            ans += pre * cur + cur * (cur - 1) // 2     # SAI: thêm cặp cùng hàng
            pre = cur
    return ans

print("bản sai :", wrong_same_row(examples[1][0]))
print("bản đúng:", Solution().numberOfBeams(examples[1][0]))

bản sai : 3
bản đúng: 0


**Sửa:** điều kiện 1 yêu cầu `r1 < r2` — hai thiết bị cùng hàng không có tia (ví dụ 2 cho đáp án 0).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đọc kỹ chữ "**for each row i where r1 < i < r2, no devices**" ⇒ chỉ **hàng không rỗng liên tiếp** mới tương tác. Phần tử "trong suốt" (hàng rỗng) ⇒ lọc bỏ.
- "Số cặp giữa hai nhóm" ⇒ **nhân**. "Số cặp trong một nhóm" ⇒ `k(k−1)/2`.
- Bài kiểu "chỉ tương tác với phần tử hợp lệ gần nhất" ⇒ quét một lượt với biến `pre`/`last`.
- Chuỗi nhị phân: dùng `str.count` thay cho vòng lặp tay để nhanh hơn nhiều trong Python.
- Bài cùng họ (quy tắc nhân + quét): 1513 Number of Substrings With Only 1s, 2063 Vowels of All Substrings, 1588.

## 15. Biến thể của bài

### 15.1 Biến thể A — Tia **ngang**: giữa hai cột không rỗng liên tiếp

Chỉ cần đếm theo cột. Dùng `zip(*bank)` để chuyển vị ma trận chuỗi.

In [28]:
def beams_by_columns(bank):
    ans = pre = 0
    for col in zip(*bank):
        cur = col.count("1")
        if cur:
            ans += pre * cur; pre = cur
    return ans

print(beams_by_columns(examples[0][0]))
print(list(zip(*["011", "100"])))

7
[('0', '1'), ('1', '0'), ('1', '0')]


### 15.2 Biến thể B — Có **tường chặn** `'#'`

Thêm ký tự `'#'` vào bản đồ. Quy ước của biến thể: một hàng **không có thiết bị nhưng có ít nhất một `'#'`** là "hàng tường" — nó chặn mọi tia đi xuyên qua. Hàng có thiết bị vẫn xử lý như bài gốc (nó bắn tia với hàng không rỗng kề trên và kề dưới).

Thay đổi duy nhất trong thuật toán quét: khi gặp hàng tường thì **quên** hàng không rỗng phía trên (`pre = 0`). Ta kiểm chứng bằng brute force viết theo đúng định nghĩa.

In [29]:
def beams_with_walls(bank):
    ans = pre = 0
    for row in bank:
        cur = row.count("1")
        if cur:
            ans += pre * cur; pre = cur
        elif "#" in row:
            pre = 0                       # tường: không tia nào đi qua
    return ans

def beams_with_walls_brute(bank):
    devices = [(r, c) for r, row in enumerate(bank) for c, ch in enumerate(row) if ch == "1"]
    cnt = 0
    for (r1, _), (r2, _) in combinations(devices, 2):
        if r1 != r2 and all(bank[i].count("1") == 0 and "#" not in bank[i] for i in range(min(r1, r2) + 1, max(r1, r2))):
            cnt += 1
    return cnt

b = ["11", "#0", "10", "00", "11"]
print(beams_with_walls(b), beams_with_walls_brute(b))
rng = random.Random(152)
for _ in range(800):
    b = ["".join(rng.choice("0001#") for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 7))]
    assert beams_with_walls(b) == beams_with_walls_brute(b)
print("biến thể tường: khớp brute force")

2 2
biến thể tường: khớp brute force


### 15.3 Biến thể C — Đáp án modulo và truy vấn bật/tắt thiết bị

Nếu có `q` truy vấn "bật/tắt ô `(r, c)`" và sau mỗi truy vấn hỏi số tia, ta có thể duy trì `counts` và tập các hàng không rỗng (sắp xếp, ví dụ `SortedList`) — mỗi truy vấn chỉ ảnh hưởng tích của hàng đó với hàng không rỗng kề trên/dưới. Dưới đây là bản đơn giản O(m) mỗi truy vấn (đủ để minh họa và kiểm chứng):

In [30]:
class BankBeams:
    def __init__(self, bank):
        self.grid = [list(r) for r in bank]
        self.counts = [r.count("1") for r in bank]
    def toggle(self, r, c):
        if self.grid[r][c] == "1":
            self.grid[r][c] = "0"; self.counts[r] -= 1
        else:
            self.grid[r][c] = "1"; self.counts[r] += 1
    def beams(self):
        ans = pre = 0
        for cur in self.counts:
            if cur: ans += pre * cur; pre = cur
        return ans

bb = BankBeams(examples[0][0])
print(bb.beams()); bb.toggle(2, 1); bb.toggle(2, 3); print(bb.beams())   # hàng 2 thành rỗng: 3*1

8
3


### 15.4 Biến thể D — Đếm **số thiết bị có ít nhất một tia**

Thiết bị có tia ⇔ hàng của nó không rỗng **và** có ít nhất một hàng không rỗng khác (khi đó nó nối với hàng kề trên hoặc dưới). Vậy đáp án = tổng thiết bị nếu có ≥ 2 hàng không rỗng, ngược lại 0.

In [31]:
def devices_with_beam(bank):
    nz = [r.count("1") for r in bank if "1" in r]
    return sum(nz) if len(nz) >= 2 else 0

def devices_with_beam_brute(bank):
    devs = [(r, c) for r, row in enumerate(bank) for c, ch in enumerate(row) if ch == "1"]
    ok = set()
    for (r1, c1), (r2, c2) in combinations(devs, 2):
        if r1 != r2 and all("1" not in bank[i] for i in range(min(r1, r2) + 1, max(r1, r2))):
            ok.add((r1, c1)); ok.add((r2, c2))
    return len(ok)

rng = random.Random(154)
for _ in range(500):
    b = ["".join(rng.choice("0001") for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 6))]
    assert devices_with_beam(b) == devices_with_beam_brute(b)
print(devices_with_beam(examples[0][0]), devices_with_beam(examples[1][0]), "- khớp brute force")

6 0 - khớp brute force


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Cặp thiết bị | đúng định nghĩa | O(D²·mn) | O(D) | chuẩn vàng để test |
| 1b. Cặp hàng | bỏ cột, xét cặp hàng | O(mn + m³) | O(m) | bước trung gian khi suy luận |
| 2. Lọc + pairwise | bỏ hàng rỗng, nhân kề | O(mn) | O(m) | code ngắn, dễ đọc |
| 3. Một lượt `pre` | nhớ hàng không rỗng gần nhất | O(mn) | O(1) | **mặc định** |
| 4. Đếm tay | như 3, không dùng `count` | O(mn) | O(1) | input là ma trận số |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `numberOfBeams(["101", "000", "010", "111"])` là bao nhiêu?

**Bài 2 (tính tay):** Một ngân hàng có số thiết bị theo hàng là `[0, 4, 0, 0, 3, 5, 0, 2]`. Tính số tia.

**Bài 3 (điền TODO):** Viết `bt3_todo(bank)` trả về **danh sách các cặp hàng** `(r1, r2)` có tia giữa chúng (theo thứ tự tăng).

**Bài 4 (tự viết lại):** Viết `bt4_todo(bank)` tính số tia **không dùng** `str.count` và **không dùng** biến `pre` — thay vào đó dùng chỉ số hàng không rỗng trước đó `last_row`.

**Bài 5 (mini challenge):** Bắt buộc **xóa đúng một hàng** (bỏ hẳn hàng đó khỏi ngân hàng, `m ≥ 2`). Viết `bt5_todo(bank)` trả về số tia **lớn nhất** có thể đạt sau khi xóa.

In [32]:
def bt3_todo(bank):
    # TODO: danh sách (r1, r2) có tia
    pass

def bt4_todo(bank):
    # TODO: dùng last_row thay cho pre, không dùng str.count
    pass

def bt5_todo(bank):
    # TODO: xóa đúng một hàng để số tia lớn nhất
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: đếm mỗi hàng. Mức 2: `[2, 0, 1, 3]`. Mức 3: `2·1 + 1·3 = 5`.

**Bài 2.** Mức 1: bỏ số 0. Mức 2: `[4, 3, 5, 2]`. Mức 3: `12 + 15 + 10 = 37`.

**Bài 3.** Mức 1: nhớ **chỉ số** hàng không rỗng gần nhất. Mức 2: mỗi hàng không rỗng (trừ hàng đầu tiên) tạo một cặp với hàng đó. Mức 3: `if cur: if last is not None: out.append((last, i)); last = i`.

**Bài 4.** Mức 1: đếm bằng `sum(ch == '1' for ch in row)`. Mức 2: `last_row = -1` ban đầu. Mức 3: `ans += cnt(bank[last_row]) * cur` — nhưng đếm lại hàng cũ tốn thêm; lưu `counts` vào list trước.

**Bài 5.** Mức 1: xóa hàng rỗng không đổi gì; xóa hàng không rỗng `r_k` thì mất `c_{k−1}c_k + c_k c_{k+1}` và thêm `c_{k−1}c_{k+1}`. Mức 2: tính delta cho mỗi `k` trong O(1). Mức 3: đáp án = base + max(delta) với delta của hàng rỗng là 0 (nếu có hàng rỗng).

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `5`. **Bài 2:** `37`.

In [33]:
assert Solution().numberOfBeams(["101", "000", "010", "111"]) == 5
c = [0, 4, 0, 0, 3, 5, 0, 2]; nz = [x for x in c if x]
assert sum(a * b for a, b in pairwise(nz)) == 37

def bt3(bank):
    out = []; last = None
    for i, row in enumerate(bank):
        if "1" in row:
            if last is not None: out.append((last, i))
            last = i
    return out

def bt4(bank):
    counts = [sum(ch == "1" for ch in row) for row in bank]
    ans = 0; last_row = -1
    for i, cur in enumerate(counts):
        if cur:
            if last_row != -1: ans += counts[last_row] * cur
            last_row = i
    return ans

def bt5(bank):
    counts = [r.count("1") for r in bank]
    nz = [x for x in counts if x]
    base = sum(a * b for a, b in pairwise(nz))
    best = base if any(x == 0 for x in counts) else None   # xóa một hàng rỗng: không đổi
    t = len(nz)
    for k in range(t):
        left = nz[k - 1] if k > 0 else 0
        right = nz[k + 1] if k + 1 < t else 0
        val = base - left * nz[k] - nz[k] * right + left * right
        best = val if best is None else max(best, val)
    return best

def bt5_brute(bank):
    return max(Solution().numberOfBeams(bank[:i] + bank[i + 1:]) for i in range(len(bank)))

assert bt3(examples[0][0]) == [(0, 2), (2, 3)]
assert bt5(["11", "1", "11"]) == 4 and bt5(examples[0][0]) == bt5_brute(examples[0][0])
rng = random.Random(1925)
for _ in range(1500):
    m, n = rng.randint(2, 8), rng.randint(1, 5)
    bank = ["".join(rng.choice("001") for _ in range(n)) for _ in range(m)]
    assert bt4(bank) == brute_devices(bank)
    assert len(bt3(bank)) == max(0, sum(1 for r in bank if "1" in r) - 1)
    assert sum(bank[a].count("1") * bank[b].count("1") for a, b in bt3(bank)) == Solution().numberOfBeams(bank)
    assert bt5(bank) == bt5_brute(bank)
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


Ghi chú bài 5: chú ý xóa hàng **rỗng** không làm thay đổi đáp án, nên nếu có hàng rỗng thì `base` luôn là một ứng viên; nếu mọi hàng đều có thiết bị thì buộc phải xóa một hàng không rỗng.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
ans = pre = 0
for row in bank:
    cur = row.count('1')
    if cur:                 # hàng rỗng "trong suốt": bỏ qua hoàn toàn
        ans += pre * cur
        pre = cur
return ans
```

**Checklist:**
- [ ] Hàng rỗng: **không** cộng, **không** đổi `pre`.
- [ ] Chỉ nhân với hàng không rỗng **gần nhất** phía trên.
- [ ] Nhân, không cộng; không đếm cặp cùng hàng.
- [ ] Cột không quan trọng.
- [ ] Đã thử: 1 hàng, toàn hàng rỗng, hàng rỗng ở đầu/cuối/giữa.